In [15]:
import numpy as np
import tensorflow as tf
import wave, math, struct
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, Activation

In [16]:
## prepare a dummy data for simulating musical notes
notes_freqs = {
    "A":440.0, "B":493.88, "C":261.63, "D":293.66, "E":393.63,"F":349.23, "G":392.0
}

In [17]:
notes_freqs

{'A': 440.0,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [18]:
notes = list(notes_freqs.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [23]:
note_to_int = {note: i for i ,note in enumerate(notes)}
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [24]:
int_to_note = {i: note for i ,note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [25]:
raw_music_data = [notes[np.random.randint(0,7)]for i in range(1000)]
raw_music_data

['A',
 'B',
 'F',
 'C',
 'D',
 'A',
 'D',
 'D',
 'B',
 'F',
 'B',
 'G',
 'B',
 'A',
 'E',
 'D',
 'A',
 'A',
 'B',
 'C',
 'F',
 'A',
 'B',
 'A',
 'C',
 'B',
 'C',
 'B',
 'C',
 'G',
 'A',
 'D',
 'D',
 'D',
 'D',
 'G',
 'D',
 'C',
 'G',
 'E',
 'G',
 'D',
 'F',
 'B',
 'G',
 'E',
 'C',
 'A',
 'F',
 'F',
 'A',
 'D',
 'A',
 'D',
 'A',
 'B',
 'C',
 'F',
 'D',
 'B',
 'E',
 'G',
 'C',
 'F',
 'E',
 'C',
 'G',
 'C',
 'D',
 'E',
 'F',
 'A',
 'G',
 'A',
 'G',
 'D',
 'E',
 'A',
 'C',
 'E',
 'A',
 'A',
 'C',
 'G',
 'D',
 'C',
 'E',
 'A',
 'E',
 'A',
 'A',
 'C',
 'A',
 'C',
 'A',
 'E',
 'F',
 'F',
 'F',
 'F',
 'D',
 'A',
 'B',
 'G',
 'E',
 'A',
 'D',
 'G',
 'E',
 'D',
 'D',
 'F',
 'F',
 'C',
 'D',
 'G',
 'G',
 'D',
 'E',
 'E',
 'B',
 'C',
 'F',
 'D',
 'B',
 'F',
 'A',
 'G',
 'B',
 'F',
 'G',
 'F',
 'D',
 'E',
 'G',
 'A',
 'D',
 'E',
 'D',
 'D',
 'B',
 'C',
 'C',
 'F',
 'F',
 'F',
 'C',
 'E',
 'D',
 'C',
 'B',
 'D',
 'G',
 'D',
 'B',
 'C',
 'C',
 'C',
 'B',
 'F',
 'C',
 'B',
 'G',
 'D',
 'C',
 'E',
 'F'

## Data Preparation

In [26]:
seq_length = 3
network_input = []
network_output = []

for i in range(len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i: i + seq_length]
    seq_out = raw_music_data[i + seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in, '-->', seq_out)

['A', 'B', 'F'] --> C
['B', 'F', 'C'] --> D
['F', 'C', 'D'] --> A
['C', 'D', 'A'] --> D
['D', 'A', 'D'] --> D
['A', 'D', 'D'] --> B
['D', 'D', 'B'] --> F
['D', 'B', 'F'] --> B
['B', 'F', 'B'] --> G
['F', 'B', 'G'] --> B
['B', 'G', 'B'] --> A
['G', 'B', 'A'] --> E
['B', 'A', 'E'] --> D
['A', 'E', 'D'] --> A
['E', 'D', 'A'] --> A
['D', 'A', 'A'] --> B
['A', 'A', 'B'] --> C
['A', 'B', 'C'] --> F
['B', 'C', 'F'] --> A
['C', 'F', 'A'] --> B
['F', 'A', 'B'] --> A
['A', 'B', 'A'] --> C
['B', 'A', 'C'] --> B
['A', 'C', 'B'] --> C
['C', 'B', 'C'] --> B
['B', 'C', 'B'] --> C
['C', 'B', 'C'] --> G
['B', 'C', 'G'] --> A
['C', 'G', 'A'] --> D
['G', 'A', 'D'] --> D
['A', 'D', 'D'] --> D
['D', 'D', 'D'] --> D
['D', 'D', 'D'] --> G
['D', 'D', 'G'] --> D
['D', 'G', 'D'] --> C
['G', 'D', 'C'] --> G
['D', 'C', 'G'] --> E
['C', 'G', 'E'] --> G
['G', 'E', 'G'] --> D
['E', 'G', 'D'] --> F
['G', 'D', 'F'] --> B
['D', 'F', 'B'] --> G
['F', 'B', 'G'] --> E
['B', 'G', 'E'] --> C
['G', 'E', 'C'] --> A
['E', 'C',

In [28]:
n_patterns = len(network_input)
n_patterns

997

In [38]:
x =np.reshape(network_input,(n_patterns,seq_length,1))
x

array([[[0],
        [1],
        [5]],

       [[1],
        [5],
        [2]],

       [[5],
        [2],
        [3]],

       ...,

       [[1],
        [0],
        [1]],

       [[0],
        [1],
        [3]],

       [[1],
        [3],
        [3]]], shape=(997, 3, 1))

In [34]:
from keras.utils import to_categorical

In [35]:
y= to_categorical(network_output)

In [36]:
y.shape

(997, 7)

In [37]:
y

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], shape=(997, 7))

## Build the model

In [39]:
model = Sequential()
model.add(GRU(256, input_shape=(3,1)))
model.add(Dense(512, activation="relu"))
model.add(Dense(7,activation="softmax"))


C:\Users\PGCP-AI\AppData\Local\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [40]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru (GRU)                            │ (None, 256)                 │         198,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 7)                   │           3,591 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 334,087 (1.27 MB)

 Trainable params: 334,087 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.compile(loss='categorical_crossentropy',metrics=['accuracy'])


In [43]:
model.fit(x,y,epochs=1000,batch_size=10)

Epoch 1/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1505 - loss: 1.9657
Epoch 2/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1535 - loss: 1.9532
Epoch 3/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1484 - loss: 1.9472
Epoch 4/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1515 - loss: 1.9463
Epoch 5/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1444 - loss: 1.9467
Epoch 6/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1535 - loss: 1.9450
Epoch 7/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1474 - loss: 1.9439
Epoch 8/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1575 - loss: 1.9423
Epoch 9/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1745 - loss: 1.9431
Epoch 10/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1595 - loss: 1.9409
Epoch 11/1000
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1585 - loss: 1.9419
Epoch 12/1000
100/100 ━━━━━━━━

## Generate new melody sequences

In [51]:
start_index = np.random.randint(0, len(network_output))
pattern = network_input[start_index]

In [61]:
generated_melody = []
for i in range(50):
  x_input = np.reshape(pattern, (1,len(pattern),1))
  pred = model.predict(x_input)
  index = np.argmax(pred)
  result = int_to_note[index]

  generated_melody.append(result)
  pattern.append(index)
  pattern = pattern[1:len(pattern)]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━

In [62]:
generated_melody

['D',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E',
 'E']

In [63]:
###  Save this as audio file

In [64]:
with wave.open('my_music.wav','w') as wav_file:
  wav_file.setparams((1,2,44100,0,'NONE','not_compressed'))
  for note in generated_melody:
    freq=notes_freqs[note]
    num_samples = int(0.5 * 44100)

    for i in range(num_samples):
      t =float(i)/44100
      value = int(32767 * 0.5 * math.sin(2 * math.pi * freq * t))
      data = struct.pack('<h',value)
      wav_file.writeframes(data)
    